#### THIRD ATTEMPT

## Data Exploration

### OpTc Dataset
Overview
The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC&utm_source=chatgpt.com

**project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.


In [1]:
# Imports
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
import tensorflow as tf

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Libraries for models
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Libraries for evaluation
from sklearn import metrics

# Libraries for deep learning
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.layers import Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam


from pathlib import Path

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")





In [2]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Load aand preview the data

In [3]:
# Location of ready flattened parquet files in Google Drive
data_path = Path("/content/drive/MyDrive/solutions/ready")

# Get all parquet files
files = list(data_path.glob("*.parquet"))

print(f"Number of files: {len(files)}")

# Inspect one
sample = pd.read_parquet(files[0])

print(sample.shape)
sample.head()
sample.info()


Number of files: 41
(143207, 62)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143207 entries, 0 to 143206
Data columns (total 62 columns):
 #   Column               Non-Null Count   Dtype                                 
---  ------               --------------   -----                                 
 0   action               143207 non-null  object                                
 1   actorID              143207 non-null  object                                
 2   hostname             143207 non-null  object                                
 3   id                   143207 non-null  object                                
 4   object               143207 non-null  object                                
 5   objectID             143207 non-null  object                                
 6   pid                  143207 non-null  int64                                 
 7   ppid                 143207 non-null  int64                                 
 8   principal            143207 non

#### 2. EDA

In [4]:
total_rows = 0
total_malicious = 0


# Revisiting the malicious - benign distribution from before,

for file in files:
    data = pd.read_parquet(file, columns=["label"])

    total_rows += len(data)
    total_malicious += data["label"].sum()

total_benign = total_rows - total_malicious

print(f"Total events: {total_rows:,}")
print(f"Benign/unlabelled events: {total_benign:,}")
print(f"Malicious events: {total_malicious:,}")
print(f"Malicious percentage: {(total_malicious / total_rows) * 100:.3f}%")

Total events: 59,405,183
Benign/unlabelled events: 59,308,239
Malicious events: 96,944
Malicious percentage: 0.163%


### 3. Data Cleaning - Missing, Infinite and Duplicate Values

In [5]:
# Identify and calculate percentage of missing values
missing_counts = None
total_rows = 0

for file in files:

    data = pd.read_parquet(file)

    file_missing = data.isnull().sum()

    if missing_counts is None:
        missing_counts = file_missing
    else:
        missing_counts = missing_counts.add(
            file_missing,
            fill_value=0
        )

    total_rows += len(data)

missing_percentage = (
    missing_counts / total_rows * 100
).sort_values(ascending=False)

print(missing_percentage)

requesting_user        99.994997
requesting_domain      99.994997
privileges             99.994844
user_name              99.994403
requesting_logon_id    99.994112
                         ...    
objectID                0.000000
label                   0.000000
principal               0.000000
tid                     0.000000
timestamp               0.000000
Length: 62, dtype: float64


In [6]:
# drop columns which have more than 99% missing values
# I might undo this later
# But I will leave this for now, so as to manage data easily

high_missing_cols = missing_percentage[missing_percentage > 99].index.tolist()

print(f"Columns with >99% missing values: {len(high_missing_cols)}")
print(high_missing_cols)



Columns with >99% missing values: 22
['requesting_user', 'requesting_domain', 'privileges', 'user_name', 'requesting_logon_id', 'logon_id', 'task_pid', 'task_process_uuid', 'path', 'task_name', 'context_info', 'payload', 'sid', 'user', 'tgt_pid_uuid', 'type', 'value', 'data', 'key', 'new_path', 'start_time', 'end_time']


In [7]:
# Check and remove duplicate rows from each file

total_duplicates = 0

for file in files:
    data = pd.read_parquet(file)

    duplicates = data.duplicated().sum()
    total_duplicates += duplicates

    if duplicates > 0:
        data = data.drop_duplicates()
        data.to_parquet(file, index=False)

print(f"Total duplicates removed: {total_duplicates:,}")

# From results, we see that there were no duplicates

Total duplicates removed: 0


In [9]:
# Check for infinite values

numeric_cols = data.select_dtypes(include=np.number).columns

infinite_values = np.isinf(data[numeric_cols]).sum()

print(infinite_values[infinite_values > 0])

# From results, we see that there were no infinite values

Series([], dtype: int64)


In [ ]:
# Select attacked and control hosts
selected_hosts = [
    # Attacked hosts
    "sysclient0201",
    "sysclient0501",
    "sysclient0811",
    "sysclient0051",
    "sysclient0351",

    # Control hosts
    "sysclient0202",
    "sysclient0502",
    "sysclient0812",
    "sysclient0052",
    "sysclient0352"
]

# Find parquet files belonging to selected hosts
selected_files = [
    file for file in files
    if any(host in file.name.lower() for host in selected_hosts)
]

print(f"Selected files: {len(selected_files)}")

# Load only selected files
data_sample = pd.concat(
    [pd.read_parquet(file) for file in selected_files],
    ignore_index=True
)

# Apply the columns already identified during cleaning
data_sample = data_sample.drop(
    columns=high_missing_cols,
    errors="ignore"
)

data_sample = data_sample.drop(
    columns=["id", "actorID", "objectID"],
    errors="ignore"
)

# Ensure timestamp is datetime
data_sample["timestamp"] = pd.to_datetime(
    data_sample["timestamp"],
    utc=True
)

# Preserve temporal order
data_sample = data_sample.sort_values(
    ["hostname", "timestamp"]
).reset_index(drop=True)

print(f"Sample shape: {data_sample.shape}")

print("\nHosts:")
print(data_sample["hostname"].value_counts())

print("\nClass distribution:")
print(data_sample["label"].value_counts())

print("\nClass percentages:")
print(data_sample["label"].value_counts(normalize=True) * 100)

Selected files: 12


### 4. Identifier Feature Dropping

In [8]:
# I am dropping these are unique identifiers so as to avoid memorisation rather than behavioural learning.

features_to_drop = [
    "id",
    "actorID",
    "objectID"
]

# I am also dropping the columns that have over 99% missing values from the previous check
data = data.drop(columns=high_missing_cols)

### 5. Create Data Sample (with referrence to temporal order)

In [13]:
print(data["hostname"].value_counts().head(50))

print(data["hostname"].dropna().unique()[:50])

hostname
SysClient0813.systemia.com    3802955
Name: count, dtype: int64
['SysClient0813.systemia.com']


In [10]:
# A random sample might be good for the preliminary detection excercise
# However, I would wannt to create a sample that I can reuse to achieve my long term goal
# Which is predicting cyber attacks before they occur


# Ensure timestamp is in datetime format
data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)

# Select attacked and control hosts
selected_hosts = [
    # Attacked hosts
    "SysClient0201.systemia.com",
    "SysClient0501.systemia.com",
    "SysClient0811.systemia.com",
    "SysClient0051.systemia.com",
    "SysClient0351.systemia.com",

    # Control hosts
    "SysClient0202.systemia.com",
    "SysClient0502.systemia.com",
    "SysClient0812.systemia.com",
    "SysClient0052.systemia.com",
    "SysClient0352.systemia.com"
]

# Create subset while retaining all events from selected hosts
data_sample = data[
    data["hostname"].isin(selected_hosts)
].copy()

# Sort events chronologically within each host
data_sample = data_sample.sort_values(
    by=["hostname", "timestamp"]
).reset_index(drop=True)

# Check resulting dataset
print(f"Sample shape: {data_sample.shape}")

print("\nClass distribution:")
print(data_sample["label"].value_counts())

print("\nClass percentages:")
print(data_sample["label"].value_counts(normalize=True) * 100)

Sample shape: (0, 59)

Class distribution:
Series([], Name: count, dtype: int64)

Class percentages:
Series([], Name: proportion, dtype: float64)
